

```
# https://drive.google.com/file/d/1r5mg6VHQ26Cg3j_OvBIyMiG4xfNjuPoK/view?usp=sharing
```



In [1]:
# 1. Install Unsloth natively via their official pre-compiled GitHub wheel for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install essential dependencies without dependency-resolution lockups
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Install utility libraries for Kaggle integration and dataset management
!pip install kagglehub datasets transformers evaluate rouge_score bleu bert_score

In [ ]:
# ==============================================================================
# 2. ENVIRONMENT PURGE & CLEAN VRAM
# ==============================================================================
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# ==============================================================================
# 3. ENVIRONMENT SETUP & ROBUST DATA LOADING
# ==============================================================================
import json
import os
from datasets import Dataset
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from transformers import EarlyStoppingCallback

In [ ]:
def load_and_format_dataset(file_path):
    formatted_data = []
    skipped_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        keyword_profile = record.get("keyword_profile", {})

        if not jd or not keyword_profile:
            continue

        target_output = json.dumps(keyword_profile, ensure_ascii=False)

        messages = [
            {
                "role": "system",
                "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
            },
            {
                "role": "user",
                "content": f"Extract the keyword profile from this job description:\n\n{jd}"
            },
            {
                "role": "assistant",
                "content": target_output
            }
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)


def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [ ]:
INPUT_FILE = "/content/era_match_keywords.jsonl"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "outputs/jd_to_keywords_model"
MAX_SEQ_LENGTH = 4048
LOAD_IN_4BIT = True

raw_dataset = load_and_format_dataset(INPUT_FILE)
dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

train_dataset = train_dataset.remove_columns(["messages"])
eval_dataset = eval_dataset.remove_columns(["messages"])

In [ ]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    per_device_eval_batch_size = 2,
    eval_accumulation_steps = 1,
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    output_dir = "outputs",
    remove_unused_columns = True,
    max_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text",

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    processing_class = tokenizer,
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)

In [ ]:
trainer_stats = trainer.train()
best_checkpoint_path = trainer.state.best_model_checkpoint

if best_checkpoint_path is not None:
    model.from_pretrained(model, best_checkpoint_path)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -r "keywords_model.zip" "outputs/jd_to_keywords_model"

Successfully loaded 5014 records. (Skipped: 0 rows)
==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

...

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Validation Inference ---


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Extracted Profile Target Output:
 {"primary": ["Senior DevOps Engineer", "Kubernetes", "Terraform", "AWS Infrastructure", "Gitlab CI/CD", "Prometheus", "Grafana"], "secondary": [], "tertiary": []}
Computing complete evaluation metrics suite...


 12%|█▏        | 29/251 [15:53<1:53:31, 30.68s/it]Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print("\n--- Running Validation Inference ---")
FastLanguageModel.for_inference(model)

sample_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

test_messages = [
    {
        "role": "system",
        "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
    },
    {
        "role": "user",
        "content": f"Extract the keyword profile from this job description:\n\n{sample_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
    top_p = 0.9,
    pad_token_id = tokenizer.eos_token_id
)

generated_output = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Extracted Profile Target Output:\n", generated_output)
print("="*60)

In [ ]:
# import evaluate
# from tqdm import tqdm

# print("Computing complete evaluation metrics suite...")
# rouge_metric = evaluate.load("rouge")
# bleu_metric = evaluate.load("bleu")
# bertscore_metric = evaluate.load("bertscore")

# generated_predictions = []
# ground_truth_references = []

# with torch.no_grad():
#     for sample in tqdm(eval_dataset):
#         full_text = sample['text']

#         parts = full_text.split("<|im_start|>assistant\n")
#         if len(parts) < 2: continue

#         prompt_part = parts[0] + "<|im_start|>assistant\n"
#         gold_truth = parts[1].replace("<|im_end|>\n", "").strip()

#         eval_inputs = tokenizer(prompt_part, return_tensors="pt").input_ids.to("cuda")

#         eval_outputs = model.generate(
#             input_ids = eval_inputs,
#             max_new_tokens = 512,
#             use_cache = True,
#             do_sample = False,
#             pad_token_id = tokenizer.eos_token_id
#         )

#         generated_response = tokenizer.decode(eval_outputs[0][len(eval_inputs[0]):], skip_special_tokens=True).strip()

#         generated_predictions.append(generated_response)
#         ground_truth_references.append(gold_truth)

# rouge_results = rouge_metric.compute(predictions=generated_predictions, references=ground_truth_references)
# bleu_results = bleu_metric.compute(predictions=generated_predictions, references=[[ref] for ref in ground_truth_references])
# bertscore_results = bertscore_metric.compute(predictions=generated_predictions, references=ground_truth_references, lang="en")

# avg_bert_f1 = sum(bertscore_results['f1']) / len(bertscore_results['f1'])

# print("\n" + "="*50)
# print("              FINAL EVALUATION REPORT              ")
# print("="*50)
# print(f"1. ROUGE-1 (Keyword Match):      {rouge_results['rouge1']:.4f}")
# print(f"2. ROUGE-2 (Phrase Match):       {rouge_results['rouge2']:.4f}")
# print(f"3. ROUGE-L (Structure Match):    {rouge_results['rougeL']:.4f}")
# print(f"4. BLEU Score (Fluency):         {bleu_results['bleu']:.4f}")
# print(f"5. BERTScore F1 (Semantic Match):{avg_bert_f1:.4f}")
# print("="*50)